# FastF1 API - Exploratory Data Analysis

This notebook explores the FastF1 API to understand:
- Available data types and structures
- How to access different datasets
- What information is available for predictions

In [ ]:
import fastf1
import pandas as pd
import numpy as np

# Enable cache for faster data loading
fastf1.Cache.enable_cache('cache')

NotADirectoryError: Cache directory does not exist! Please check for typos or create it first.

## 1. Session Data Overview

Let's start by loading a recent session and exploring what data is available.

In [ ]:
# Load a recent session (e.g., Abu Dhabi 2024 Race)
session = fastf1.get_session(2024, 'Abu Dhabi', 'R')
session.load()

print(f"Event: {session.event['EventName']}")
print(f"Session: {session.name}")
print(f"Date: {session.event['EventDate']}")

## 2. Results Data

Race results including positions, points, and status.

In [ ]:
# Get race results
results = session.results
print("Available columns in results:")
print(results.columns.tolist())
print("\nFirst few results:")
results[['DriverNumber', 'BroadcastName', 'Abbreviation', 'TeamName', 'Position', 'GridPosition', 'Points', 'Status']]

## 3. Lap Data

Detailed lap-by-lap information for all drivers.

In [ ]:
# Get all laps
laps = session.laps
print("Available columns in laps:")
print(laps.columns.tolist())
print(f"\nTotal number of laps: {len(laps)}")
print("\nSample lap data:")
laps.head()

In [ ]:
# Analyze lap times for a specific driver
driver_abbr = 'VER'  # Change to any driver
driver_laps = laps.pick_driver(driver_abbr)
print(f"Laps for {driver_abbr}:")
driver_laps[['LapNumber', 'LapTime', 'Compound', 'TyreLife', 'Stint', 'TrackStatus']].head(10)

## 4. Telemetry Data

Detailed telemetry for specific laps (speed, throttle, brake, etc.).

In [ ]:
# Get telemetry for a specific lap
fastest_lap = driver_laps.pick_fastest()
telemetry = fastest_lap.get_telemetry()

print("Available telemetry columns:")
print(telemetry.columns.tolist())
print("\nSample telemetry data:")
telemetry.head()

## 5. Weather Data

Track and weather conditions during the session.

In [ ]:
# Get weather data
weather = session.weather_data
print("Available weather columns:")
print(weather.columns.tolist())
print("\nWeather data summary:")
weather.describe()

## 6. Event Schedule

Get information about the season calendar.

In [ ]:
# Get 2024 season schedule
schedule = fastf1.get_event_schedule(2024)
print("Available schedule columns:")
print(schedule.columns.tolist())
print("\n2024 Season Events:")
schedule[['RoundNumber', 'EventName', 'EventDate', 'Country', 'Location']]

## 7. Practice and Qualifying Data

Explore practice and qualifying sessions for additional insights.

In [ ]:
# Load qualifying session
quali = fastf1.get_session(2024, 'Abu Dhabi', 'Q')
quali.load()

# Get qualifying results
quali_results = quali.results
print("Qualifying Results:")
quali_results[['Abbreviation', 'TeamName', 'Q1', 'Q2', 'Q3', 'Position']]

## 8. Driver and Team Information

Access driver and team metadata.

In [ ]:
# Get driver information from results
drivers = results[['DriverNumber', 'BroadcastName', 'Abbreviation', 'TeamName', 'TeamColor', 'HeadshotUrl']]
print("Driver Information:")
drivers

## 9. Key Statistics

Calculate some useful statistics for predictions.

In [ ]:
# Average lap times per driver
avg_lap_times = laps.groupby('Driver')['LapTime'].mean().sort_values()
print("Average Lap Times (excluding outliers):")
# Filter out pit laps and slow laps
clean_laps = laps[laps['IsPersonalBest'] | (laps['LapTime'] < laps['LapTime'].quantile(0.75))]
avg_lap_times_clean = clean_laps.groupby('Driver')['LapTime'].mean().sort_values()
print(avg_lap_times_clean)

In [ ]:
# Tire compound usage
tire_usage = laps.groupby(['Driver', 'Compound']).size().unstack(fill_value=0)
print("Tire Compound Usage by Driver:")
tire_usage

## 10. Data for Multiple Races

Example of collecting data across multiple races for pattern analysis.

In [ ]:
# Collect results from last 3 races
recent_events = ['Las Vegas', 'Qatar', 'Abu Dhabi']
all_results = []

for event_name in recent_events:
    try:
        race = fastf1.get_session(2024, event_name, 'R')
        race.load()
        results_df = race.results[['Abbreviation', 'TeamName', 'Position', 'GridPosition', 'Points']].copy()
        results_df['Event'] = event_name
        all_results.append(results_df)
    except Exception as e:
        print(f"Could not load {event_name}: {e}")

if all_results:
    combined_results = pd.concat(all_results, ignore_index=True)
    print("Combined Recent Results:")
    print(combined_results)

## Summary

Key data available for predictions:
- **Results**: Final positions, grid positions, points, status
- **Laps**: Lap times, tire compounds, stint information, track status
- **Telemetry**: Speed, throttle, brake, gear, DRS (per lap)
- **Weather**: Track temp, air temp, humidity, rainfall, wind
- **Schedule**: Event information, dates, locations
- **Qualifying**: Q1/Q2/Q3 times and positions

This data can be used to build features for race predictions!